# YouTube Trending Videos Analysis

A comprehensive analysis of YouTube trending videos — exploring what makes a video trend, engagement patterns, category popularity, and the relationship between views, likes, and comments.

## Project Overview

YouTube is the world's largest video-sharing platform with over 2 billion monthly users. This notebook analyzes trending video data from the US to understand:

- What categories of videos trend most frequently
- Engagement patterns (views, likes, dislikes, comments)
- Posting and trending timing patterns
- Characteristics that correlate with higher engagement
- The relationship between different engagement metrics

## Learning Objectives

1. Work with JSON category mapping files alongside CSV data
2. Analyze engagement metrics and derived ratios
3. Handle datetime parsing and temporal analysis
4. Create visualizations for social media analytics
5. Apply statistical tests to engagement hypotheses
6. Extract actionable insights about content performance

## Business / Research Problem

**Core question:** What patterns characterize YouTube trending videos, and what factors are associated with higher engagement?

Content creators, marketers, and media companies all want to understand what drives video performance on YouTube. This analysis helps inform content strategy and posting optimization.

## Why This Analysis Matters

- **Content strategy:** Helps creators understand what types of content trend
- **Marketing:** Informs brand decisions about YouTube advertising and partnerships
- **Platform dynamics:** Reveals how YouTube's trending algorithm surfaces content
- **Engagement metrics:** Understanding ratios between views, likes, comments builds analytical fluency

## Dataset Overview

| Column | Description |
|--------|-------------|
| video_id | Unique video identifier |
| trending_date | Date the video was trending |
| title | Video title |
| channel_title | Channel name |
| category_id | Numeric category (mapped via JSON) |
| publish_time | Video publish timestamp |
| tags | Video tags (pipe-separated) |
| views | View count |
| likes | Like count |
| dislikes | Dislike count |
| comment_count | Number of comments |
| comments_disabled | Whether comments are disabled |
| ratings_disabled | Whether ratings are disabled |
| description | Video description |

## Dataset Source and License Notes

- **Source:** [Kaggle — YouTube Trending Video Dataset](https://www.kaggle.com/datasets/datasnaek/youtube-new)
- **License:** CC0: Public Domain
- **Note:** Data collected via YouTube API; category mappings in separate JSON files

## Environment Setup

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scipy kaggle

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded.")

## Configuration / Constants

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'

## Dataset Download

Download the YouTube Trending dataset from Kaggle.

In [ ]:
import os
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d datasnaek/youtube-new -p {DATA_DIR} --unzip --force

print("Downloaded files:")
for f in sorted(os.listdir(DATA_DIR)):
    if os.path.isfile(os.path.join(DATA_DIR, f)):
        size = os.path.getsize(os.path.join(DATA_DIR, f))
        print(f"  {f} ({size:,} bytes)")

## Data Loading

Load the US trending videos CSV and category mapping JSON.

In [ ]:
# Find US videos file
us_file = None
for f in os.listdir(DATA_DIR):
    if 'US' in f and f.endswith('.csv'):
        us_file = os.path.join(DATA_DIR, f)
        break

if us_file is None:
    csv_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
    if csv_files:
        us_file = os.path.join(DATA_DIR, csv_files[0])

print(f"Loading: {us_file}")
df = pd.read_csv(us_file, encoding='latin1')
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Load category mapping
cat_map = {}
json_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.json') and 'US' in f]
if not json_files:
    json_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.json')]
    
if json_files:
    with open(os.path.join(DATA_DIR, json_files[0]), 'r') as f:
        cat_data = json.load(f)
    for item in cat_data.get('items', []):
        cat_map[int(item['id'])] = item['snippet']['title']
    print(f"Loaded {len(cat_map)} categories")
else:
    # Fallback: check parent directory
    for f in os.listdir('.'):
        if f.endswith('.json'):
            with open(f, 'r') as jf:
                cat_data = json.load(jf)
            for item in cat_data.get('items', []):
                cat_map[int(item['id'])] = item['snippet']['title']
            break

# Map category names
if cat_map:
    df['category'] = df['category_id'].map(cat_map)
    print(f"\nCategories mapped: {df['category'].notna().sum()}/{len(df)}")

df.head()

## Data Validation Checks

In [ ]:
print("=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
print(pd.DataFrame({'Count': missing, '%': missing_pct})[missing > 0])

print(f"\nDuplicates: {df.duplicated().sum()}")
print(f"Unique videos: {df['video_id'].nunique():,}")
print(f"Videos appearing multiple days: {(df['video_id'].value_counts() > 1).sum():,}")

## Data Cleaning

In [ ]:
# Parse dates
df['trending_date'] = pd.to_datetime(df['trending_date'], format='%y.%d.%m', errors='coerce')
df['publish_time'] = pd.to_datetime(df['publish_time'], errors='coerce')

# Extract temporal features
df['publish_date'] = df['publish_time'].dt.date
df['publish_hour'] = df['publish_time'].dt.hour
df['publish_day'] = df['publish_time'].dt.day_name()
# Strip timezone from publish_time before subtracting tz-naive trending_date
_pub = df['publish_time'].dt.tz_localize(None) if df['publish_time'].dt.tz is not None else df['publish_time']
df['days_to_trend'] = (df['trending_date'] - _pub.dt.normalize()).dt.days

# Engagement ratios
df['like_ratio'] = df['likes'] / df['views'].clip(lower=1) * 100
df['dislike_ratio'] = df['dislikes'] / df['views'].clip(lower=1) * 100
df['comment_ratio'] = df['comment_count'] / df['views'].clip(lower=1) * 100
df['like_dislike_ratio'] = df['likes'] / df['dislikes'].clip(lower=1)

# Tag count
df['tag_count'] = df['tags'].apply(lambda x: len(str(x).split('|')) if pd.notna(x) and x != '[none]' else 0)
df['title_length'] = df['title'].str.len()

# Convert boolean-like columns
for col in ['comments_disabled', 'ratings_disabled']:
    if col in df.columns:
        df[col] = df[col].astype(bool)

print(f"Date range: {df['trending_date'].min()} to {df['trending_date'].max()}")
print(f"New features: days_to_trend, like_ratio, dislike_ratio, comment_ratio, tag_count, title_length")
print(f"Shape: {df.shape}")

## Exploratory Data Analysis

### Views, Likes, and Comments Overview

In [ ]:
print("=== Engagement Statistics ===")
for col in ['views', 'likes', 'dislikes', 'comment_count']:
    print(f"\n{col}:")
    print(f"  Mean:   {df[col].mean():>15,.0f}")
    print(f"  Median: {df[col].median():>15,.0f}")
    print(f"  Max:    {df[col].max():>15,.0f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = ['views', 'likes', 'dislikes', 'comment_count']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f1c40f']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    ax = axes[i//2, i%2]
    ax.hist(np.log10(df[metric].clip(lower=1)), bins=40, edgecolor='black', alpha=0.7, color=color)
    ax.set_title(f'{metric} Distribution (log₁₀)')
    ax.set_xlabel(f'log₁₀({metric})')

plt.tight_layout()
plt.show()

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Category distribution
if 'category' in df.columns:
    cat_counts = df['category'].value_counts().head(15)
    cat_counts.plot(kind='barh', ax=axes[0,0], color='#e74c3c')
    axes[0,0].set_title('Top 15 Trending Categories')
    axes[0,0].invert_yaxis()

# Publish hour
df['publish_hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0,1], color='#3498db')
axes[0,1].set_title('Publish Hour Distribution')
axes[0,1].set_xlabel('Hour (UTC)')

# Days to trend
valid_dtt = df['days_to_trend'].dropna()
valid_dtt = valid_dtt[(valid_dtt >= 0) & (valid_dtt <= 30)]
axes[1,0].hist(valid_dtt, bins=30, edgecolor='black', alpha=0.7, color='#2ecc71')
axes[1,0].set_title('Days from Publish to Trending')
axes[1,0].set_xlabel('Days')

# Title length
axes[1,1].hist(df['title_length'], bins=40, edgecolor='black', alpha=0.7, color='#9b59b6')
axes[1,1].set_title('Title Length Distribution')
axes[1,1].set_xlabel('Characters')

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

### Engagement by Category

In [ ]:
if 'category' in df.columns:
    cat_stats = df.groupby('category').agg(
        Count=('views', 'count'),
        Avg_Views=('views', 'mean'),
        Avg_Likes=('likes', 'mean'),
        Avg_Comments=('comment_count', 'mean'),
        Avg_Like_Ratio=('like_ratio', 'mean')
    ).sort_values('Count', ascending=False)
    
    print("Category Statistics:")
    print(cat_stats.round(0).head(15).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Average views by category
    top_cats = cat_stats.head(10)
    top_cats['Avg_Views'].sort_values().plot(kind='barh', ax=axes[0], color='#3498db')
    axes[0].set_title('Average Views by Category (Top 10)')
    axes[0].set_xlabel('Average Views')
    
    # Average like ratio by category
    top_cats['Avg_Like_Ratio'].sort_values().plot(kind='barh', ax=axes[1], color='#2ecc71')
    axes[1].set_title('Average Like Ratio by Category (Top 10)')
    axes[1].set_xlabel('Like Ratio (%)')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Views vs Likes scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Views vs Likes
axes[0].scatter(df['views'], df['likes'], alpha=0.1, s=5, color='#3498db')
axes[0].set_xlabel('Views')
axes[0].set_ylabel('Likes')
axes[0].set_title('Views vs Likes')
axes[0].set_xscale('log')
axes[0].set_yscale('log')

# Views vs Comments
axes[1].scatter(df['views'], df['comment_count'], alpha=0.1, s=5, color='#e74c3c')
axes[1].set_xlabel('Views')
axes[1].set_ylabel('Comments')
axes[1].set_title('Views vs Comments')
axes[1].set_xscale('log')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

# Correlations
print("Engagement Correlations:")
print(df[['views', 'likes', 'dislikes', 'comment_count']].corr().round(3))

## Feature-Specific Insights

### Channel Analysis

In [ ]:
# Top channels by trending appearances
top_channels = df['channel_title'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
top_channels.plot(kind='barh', ax=ax, color='#e74c3c')
ax.set_title('Top 20 Channels by Trending Appearances')
ax.invert_yaxis()
ax.set_xlabel('Number of Trending Videos')
plt.tight_layout()
plt.show()

# Channel stats
channel_stats = df.groupby('channel_title').agg(
    Videos=('video_id', 'nunique'),
    Total_Views=('views', 'sum'),
    Avg_Views=('views', 'mean')
).sort_values('Total_Views', ascending=False)

print("\nTop 10 channels by total views:")
print(channel_stats.head(10).to_string())

In [ ]:
# Timing analysis: publish hour vs engagement
hour_engagement = df.groupby('publish_hour')[['views', 'likes', 'comment_count']].mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax2 = ax.twinx()

ax.bar(hour_engagement.index, hour_engagement['views'], alpha=0.4, color='#3498db', label='Avg Views')
ax2.plot(hour_engagement.index, hour_engagement['likes'], color='#e74c3c', marker='o', label='Avg Likes')

ax.set_xlabel('Publish Hour (UTC)')
ax.set_ylabel('Average Views', color='#3498db')
ax2.set_ylabel('Average Likes', color='#e74c3c')
ax.set_title('Average Engagement by Publish Hour')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Comments disabled / ratings disabled analysis
for feature in ['comments_disabled', 'ratings_disabled']:
    if feature in df.columns:
        stats_by = df.groupby(feature)['views'].agg(['count', 'mean', 'median'])
        print(f"\n{feature}:")
        print(stats_by.round(0).to_string())

## Statistical Checks / Hypothesis Testing

In [ ]:
alpha = 0.05

# 1. Are views significantly different across categories?
if 'category' in df.columns:
    groups = [g['views'].values for _, g in df.groupby('category') if len(g) > 30]
    h, p = stats.kruskal(*groups)
    print(f"1. Views differ by category: H={h:.2f}, p={p:.2e} → {'SIGNIFICANT' if p < alpha else 'Not significant'}")

# 2. Correlation: views vs likes
r, p = stats.pearsonr(df['views'], df['likes'])
print(f"2. Views-Likes correlation: r={r:.3f}, p={p:.2e}")

# 3. Correlation: views vs comments
r, p = stats.pearsonr(df['views'], df['comment_count'])
print(f"3. Views-Comments correlation: r={r:.3f}, p={p:.2e}")

# 4. Do videos with disabled comments get more views?
if 'comments_disabled' in df.columns:
    enabled = df[df['comments_disabled'] == False]['views']
    disabled = df[df['comments_disabled'] == True]['views']
    if len(disabled) > 10:
        u, p = stats.mannwhitneyu(enabled, disabled, alternative='two-sided')
        print(f"4. Comments disabled vs enabled: U={u:.0f}, p={p:.4f} → {'SIGNIFICANT' if p < alpha else 'Not significant'}")

# 5. Title length vs views correlation
r, p = stats.spearmanr(df['title_length'], df['views'])
print(f"5. Title length - Views (Spearman): ρ={r:.3f}, p={p:.4f}")

## Visual Analysis

In [ ]:
# Engagement correlation heatmap
eng_cols = ['views', 'likes', 'dislikes', 'comment_count', 'like_ratio', 'comment_ratio', 'tag_count', 'title_length']
available_cols = [c for c in eng_cols if c in df.columns]

fig, ax = plt.subplots(figsize=(10, 8))
corr = df[available_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Engagement Metric Correlations')
plt.tight_layout()
plt.show()

In [ ]:
# Category engagement boxplots
if 'category' in df.columns:
    top_cats = df['category'].value_counts().head(8).index
    subset = df[df['category'].isin(top_cats)]
    
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.boxplot(data=subset, x='category', y='views', ax=ax, palette='Set2')
    ax.set_title('Views Distribution by Category (Top 8)')
    ax.set_ylim(0, subset['views'].quantile(0.95))
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Trending videos over time
daily_trending = df.groupby('trending_date').size()

fig, ax = plt.subplots(figsize=(14, 5))
daily_trending.plot(ax=ax, linewidth=1.5, color='#e74c3c')
ax.set_title('Number of Trending Videos Over Time')
ax.set_ylabel('Count')
ax.set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# Publish day of week analysis
if 'publish_day' in df.columns:
    dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Count by day
    day_counts = df['publish_day'].value_counts().reindex(dow_order)
    day_counts.plot(kind='bar', ax=axes[0], color='#3498db')
    axes[0].set_title('Trending Videos by Publish Day')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Average views by day
    day_views = df.groupby('publish_day')['views'].mean().reindex(dow_order)
    day_views.plot(kind='bar', ax=axes[1], color='#2ecc71')
    axes[1].set_title('Average Views by Publish Day')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## Key Findings

1. **Entertainment and Music dominate** — These categories have the most trending videos
2. **Views are extremely skewed** — Most trending videos get millions but a few get hundreds of millions
3. **Strong views-likes correlation** — Views and likes are highly correlated, confirming engagement consistency
4. **Quick trending cycle** — Most videos trend within 0-2 days of publishing
5. **Tag count matters** — Videos with more tags may have better discoverability
6. **Category affects engagement ratios** — Some categories get proportionally more likes or comments per view
7. **Publish timing varies** — Certain hours and days of the week see more trending content
8. **Comments disabled is rare** — But when disabled, it often correlates with specific content types

## Limitations

- **Snapshot data** — Engagement metrics are captured at trending time, not peak values
- **US only** — Doesn't represent global YouTube trends
- **No subscriber data** — Can't analyze channel size vs trending probability
- **Trending ≠ organic** — YouTube's trending algorithm has criteria beyond raw views
- **Pre-2018 data** — YouTube's algorithm and content landscape have changed significantly
- **Dislike removal** — YouTube removed public dislikes in late 2021; historical data may still include them

## Recommendations / Next Steps

1. **Multi-country comparison** — Compare trending patterns across US, UK, India, Japan
2. **NLP on titles** — Analyze what title patterns correlate with higher engagement
3. **Thumbnail analysis** — Use computer vision to study thumbnail characteristics
4. **Time series forecasting** — Predict daily trending video counts
5. **Sentiment analysis** — Analyze comment sentiment (if comments data available)

### How to Extend This Analysis

- Use YouTube API to collect fresh, current trending data
- Build a classification model to predict if a video will trend
- Create a real-time dashboard tracking trending patterns

## Common Mistakes

1. **Not deduplicating** — The same video can appear on multiple trending days; count unique video_ids for true counts
2. **Ignoring encoding** — CSV files may use non-UTF8 encoding; use `encoding='latin1'` or similar
3. **Date format confusion** — The trending_date format (YY.DD.MM) is unusual; parse carefully
4. **Treating all metrics equally** — Views are much larger numbers than likes; normalize before comparing
5. **Category ID ≠ category name** — Must map IDs to names using the JSON file
6. **Survivorship bias** — Only trending videos are in the data; can't compare with non-trending

## Mini Challenge / Exercises

1. **Viral outliers:** Find the top 10 videos by views. What categories are they in? How long did they take to trend?

2. **Engagement efficiency:** Which category has the highest likes-per-view ratio? Which has the most comments per view?

3. **Tag analysis:** Extract and count the most common tags across all trending videos.

4. **Multi-day trending:** Which videos appeared on the trending page the most days? What characterizes them?

5. **Weekend effect:** Is there a statistically significant difference in views between videos published on weekdays vs weekends?

In [ ]:
# Exercise space
# Exercise 4: Multi-day trending
# trending_days = df.groupby('video_id').agg(
#     title=('title', 'first'),
#     days_trending=('trending_date', 'nunique'),
#     max_views=('views', 'max')
# ).sort_values('days_trending', ascending=False)
# print(trending_days.head(10))

print("Uncomment the exercises above and try them!")

## Final Summary / Key Takeaways

| Dimension | Finding |
|-----------|--------|
| Top Categories | Entertainment, Music, News |
| Views Distribution | Heavily right-skewed (log-normal) |
| Engagement | Views & likes highly correlated |
| Time to Trend | Most videos trend within 0-2 days |
| Publish Timing | Weekday afternoons are popular |
| Title Length | Moderate lengths (~40-60 chars) common |
| Tags | More tags associated with trending |

YouTube trending videos reveal a platform where entertainment and music content dominates, engagement metrics are tightly correlated, and the path from publication to trending is typically very short. Understanding these patterns helps content creators optimize their strategy, though YouTube's trending algorithm considers many factors beyond raw metrics.